No,４

In [4]:
import os
import re
import tkinter as tk
from tkinter import filedialog
from pathlib import Path

拡張子ごとにフォルダ分けしたファイルの名前を「0+元のファイル名から抽出した連番数字_サブサブフォルダ名_サブフォルダ名」に変更

In [5]:
#指定したメインフォルダ内のすべてのファイル名を「0+抽出した連番数字_サブサブフォルダ名_サブフォルダ名」となるように変更


def rename_csv_files_flexible():
    # 1. フォルダ選択ダイアログの表示設定
    root = tk.Tk()
    root.attributes('-topmost', True) 
    root.withdraw()

    # 一番上にある大元のフォルダを選択
    top_dir_path = filedialog.askdirectory(title="一番上の大元フォルダを選択してください")
    
    if not top_dir_path:
        print("フォルダ選択がキャンセルされました。処理を終了します。")
        return

    top_dir = Path(top_dir_path)
    print(f"選択された大元フォルダ: {top_dir.name}\n")

    rename_count = 0
    
    # 2. rglob("*.csv") で、階層の深さに関係なく大元フォルダ内の全CSVファイルを取得
    for file_path in top_dir.rglob("*.csv"):
        original_name = file_path.name
        
        # 3. ファイルから見て「末端のフォルダ」と「末端の前のフォルダ」の名前を取得
        terminal_folder = file_path.parent.name         # 末端のフォルダ名
        pre_terminal_folder = file_path.parent.parent.name # 末端の前のフォルダ名
        
        # 4. 正規表現で「$」の後の連番数字を抽出
        match = re.search(r'\$(\d+)\.csv$', original_name)
        
        if match:
            seq_num = match.group(1) # 抽出した連番数字
            
            # 5. 新しいファイル名の作成
            # 修正箇所: 0 + 抽出した連番数字 + _ + 末端の前のフォルダ名 + _ + 末端のフォルダ名 + .csv
            new_name = f"0{seq_num}_{pre_terminal_folder}_{terminal_folder}.csv"
            new_file_path = file_path.parent / new_name
            
            # 6. ファイル名の変更を実行
            try:
                file_path.rename(new_file_path)
                print(f"変更完了: {pre_terminal_folder}/{terminal_folder}/{original_name} -> {new_name}")
                rename_count += 1
            except Exception as e:
                print(f"エラー ({original_name}): {e}")

    print(f"\n処理が完了しました。合計 {rename_count} 件のファイル名を変更しました。")

# 関数の実行
rename_csv_files_flexible()

選択された大元フォルダ: csv

変更完了: 190℃/160/160.0$10.csv -> 010_190℃_160.csv
変更完了: 190℃/160/160.0$11.csv -> 011_190℃_160.csv
変更完了: 190℃/160/160.0$2.csv -> 02_190℃_160.csv
変更完了: 190℃/160/160.0$3.csv -> 03_190℃_160.csv
変更完了: 190℃/160/160.0$4.csv -> 04_190℃_160.csv
変更完了: 190℃/160/160.0$5.csv -> 05_190℃_160.csv
変更完了: 190℃/160/160.0$6.csv -> 06_190℃_160.csv
変更完了: 190℃/160/160.0$7.csv -> 07_190℃_160.csv
変更完了: 190℃/160/160.0$8.csv -> 08_190℃_160.csv
変更完了: 190℃/160/160.0$9.csv -> 09_190℃_160.csv
変更完了: 190℃/200/200.0$10.csv -> 010_190℃_200.csv
変更完了: 190℃/200/200.0$11.csv -> 011_190℃_200.csv
変更完了: 190℃/200/200.0$2.csv -> 02_190℃_200.csv
変更完了: 190℃/200/200.0$3.csv -> 03_190℃_200.csv
変更完了: 190℃/200/200.0$4.csv -> 04_190℃_200.csv
変更完了: 190℃/200/200.0$5.csv -> 05_190℃_200.csv
変更完了: 190℃/200/200.0$6.csv -> 06_190℃_200.csv
変更完了: 190℃/200/200.0$7.csv -> 07_190℃_200.csv
変更完了: 190℃/200/200.0$8.csv -> 08_190℃_200.csv
変更完了: 190℃/200/200.0$9.csv -> 09_190℃_200.csv
変更完了: 190℃/240/240.0$10.csv -> 010_190℃_240.csv
変更完了: 

ファイル名変更後のファイルの移動と、ファイルが入っていた末端フォルダの削除

In [6]:
#ファイル名変更後、末端フォルダの削除と末端フォルダの前フォルダに名前変更後ファイルの移動
#移動後に連番の振り直し

def reorganize_resort_and_rename_001():
    # 1. フォルダ選択ダイアログの表示設定
    root = tk.Tk()
    root.attributes('-topmost', True) 
    root.withdraw()

    # 一番上にある大元のフォルダを選択
    top_dir_path = filedialog.askdirectory(title="一番上の大元フォルダを選択してください")
    
    if not top_dir_path:
        print("フォルダ選択がキャンセルされました。処理を終了します。")
        return

    top_dir = Path(top_dir_path)
    print(f"選択された大元フォルダ: {top_dir.name}\n")

    pre_term_files = {}
    folders_to_delete = set()

    # ==========================================
    # 手順1: ファイルの抽出と末端前フォルダへの移動
    # ==========================================
    for file_path in top_dir.rglob("*.csv"):
        terminal_folder = file_path.parent
        pre_terminal_folder = file_path.parent.parent
        
        if terminal_folder == top_dir or pre_terminal_folder == top_dir.parent:
            continue
            
        try:
            temp_move_path = pre_terminal_folder / f"temp_{file_path.name}"
            file_path.rename(temp_move_path)
            
            folders_to_delete.add(terminal_folder)
            
            if pre_terminal_folder not in pre_term_files:
                pre_term_files[pre_terminal_folder] = []
            pre_term_files[pre_terminal_folder].append(temp_move_path)
            
        except Exception as e:
            print(f"移動エラー ({file_path.name}): {e}")

    # ==========================================
    # 手順2: 後ろの数字によるソートと、連番の振り直し
    # ==========================================
    for pre_term_dir, files in pre_term_files.items():
        sortable_list = []
        
        for temp_path in files:
            original_name = temp_path.name.replace("temp_", "", 1)
            
            match = re.match(r'^(\d+)_(.*)_(\d+)\.csv$', original_name)
            
            if match:
                front_str = match.group(1)
                middle_str = match.group(2)
                back_str = match.group(3)
                
                front_num = int(front_str)
                back_num = int(back_str)
                
                sortable_list.append({
                    'back_num': back_num,
                    'front_num': front_num,
                    'front_str': front_str,
                    'middle_str': middle_str,
                    'back_str': back_str,
                    'temp_path': temp_path
                })
            else:
                print(f"警告: ファイル名形式が一致しません。リネームをスキップします: {original_name}")
                temp_path.rename(pre_term_dir / original_name)

        sortable_list.sort(key=lambda x: (x['back_num'], x['front_num']))

        # === 変更箇所：1からスタートし、3桁の0埋めに設定 ===
        # enumerate(..., start=1) でカウントを1から開始します
        for i, item in enumerate(sortable_list, start=1):
            # :03d を指定することで「001」「002」...という3桁のフォーマットになります
            new_front = f"{i:03d}"
            
            new_name = f"{new_front}_{item['middle_str']}_{item['back_str']}.csv"
            final_path = pre_term_dir / new_name
            
            try:
                item['temp_path'].rename(final_path)
                print(f"整理完了: {pre_term_dir.name}/{new_name}")
            except Exception as e:
                print(f"リネームエラー: {e}")

    # ==========================================
    # 手順3: 空になった末端フォルダの削除
    # ==========================================
    print("\n--- フォルダの整理を開始します ---")
    deleted_folder_count = 0
    
    for folder in folders_to_delete:
        ds_store = folder / ".DS_Store"
        if ds_store.exists():
            ds_store.unlink()
            
        try:
            folder.rmdir()
            deleted_folder_count += 1
        except OSError:
            print(f"スキップ: {folder.name} (内部にCSV以外のファイルが残っているため保持しました)")

    print(f"\nすべての処理が完了しました！")
    print(f"{deleted_folder_count} 個の不要な末端フォルダを削除しました。")

# 関数の実行
reorganize_resort_and_rename_001()

選択された大元フォルダ: csv

整理完了: 190℃/001_190℃_160.csv
整理完了: 190℃/002_190℃_160.csv
整理完了: 190℃/003_190℃_160.csv
整理完了: 190℃/004_190℃_160.csv
整理完了: 190℃/005_190℃_160.csv
整理完了: 190℃/006_190℃_160.csv
整理完了: 190℃/007_190℃_160.csv
整理完了: 190℃/008_190℃_160.csv
整理完了: 190℃/009_190℃_160.csv
整理完了: 190℃/010_190℃_160.csv
整理完了: 190℃/011_190℃_200.csv
整理完了: 190℃/012_190℃_200.csv
整理完了: 190℃/013_190℃_200.csv
整理完了: 190℃/014_190℃_200.csv
整理完了: 190℃/015_190℃_200.csv
整理完了: 190℃/016_190℃_200.csv
整理完了: 190℃/017_190℃_200.csv
整理完了: 190℃/018_190℃_200.csv
整理完了: 190℃/019_190℃_200.csv
整理完了: 190℃/020_190℃_200.csv
整理完了: 190℃/021_190℃_240.csv
整理完了: 190℃/022_190℃_240.csv
整理完了: 190℃/023_190℃_240.csv
整理完了: 190℃/024_190℃_240.csv
整理完了: 190℃/025_190℃_240.csv
整理完了: 190℃/026_190℃_240.csv
整理完了: 190℃/027_190℃_240.csv
整理完了: 190℃/028_190℃_240.csv
整理完了: 190℃/029_190℃_240.csv
整理完了: 190℃/030_190℃_240.csv
整理完了: 190℃/031_190℃_280.csv
整理完了: 190℃/032_190℃_280.csv
整理完了: 190℃/033_190℃_280.csv
整理完了: 190℃/034_190℃_280.csv
整理完了: 190℃/035_190℃_280.csv
整理